# Module 2 • Class 3 — Data Visualization (Guided Companion Notebook)

**Level 1 — guided / classroom companion**

This version is for students who need more scaffolding.  
The **theme is the same** as the other versions:

- Superstore dataset
- Matplotlib + Seaborn
- build charts
- read patterns
- write short interpretations

## What we will do
1. Histogram — distribution of **Sales**
2. Bar charts — compare **Category** and **Region**
3. Scatter plot — relationship between **Sales** and **Profit**
4. Boxplot + IQR — outliers in **Profit**
5. Correlation heatmap — relationships between numeric columns
6. Monthly line chart — trend over time
7. Reflection — what the charts mean for business

**How to use this notebook in class**
- Read the short explanation
- Run the ready example
- Do the tiny TODO
- Write 1–2 sentence interpretation


In [ ]:
import pandas as pd           # Load pandas library for data manipulation
import numpy as np            # Load numpy library for numerical operations
import matplotlib.pyplot as plt # Load matplotlib for creating static visualizations
import seaborn as sns          # Load seaborn for statistical data visualization
from pathlib import Path       # Load Path class for object-oriented filesystem paths

sns.set_style("whitegrid")     # Set the aesthetic style of the plots to whitegrid
plt.rcParams["figure.figsize"] = (10, 6) # Set default size of figures in inches
plt.rcParams["font.size"] = 11         # Set default font size for plot elements

## Dataset loading — choose ONE source

Use the same loading block in all three notebook levels.

**Option 1 — `preferred`**: use your cleaned file from Module 2 Class 2: `superstore_cleaned.csv`. This is the best option because Class 3 continues the Class 2 data-preparation work.

**Option 2 — `manual_upload`**: upload a CSV manually from your computer. Use this if your cleaned file is not already in the Colab runtime.

**Option 3 — `kaggle`**: download the original Superstore dataset from Kaggle with `kagglehub`. Use this as an external fallback if you do not have the cleaned CSV.

In the next code cell, change `DATA_SOURCE` to one of: `"preferred"`, `"manual_upload"`, or `"kaggle"`.


In [ ]:
DATA_SOURCE = "kaggle"  #@param ["preferred", "manual_upload", "kaggle"]                                # Define data source source

def read_csv_robust(file_path):                                                                      # Define robust CSV reader
    """Read a CSV with common encodings used by the Superstore dataset."""
    file_path = Path(file_path)                                                                       # Convert to path object
    for encoding in ["utf-8", "latin-1", "cp1252"]:                                                   # Loop common encodings
        try:
            return pd.read_csv(file_path, encoding=encoding)                                          # Try reading file
        except UnicodeDecodeError:                                                                    # Catch decoding errors
            continue                                                                                  # Try next encoding
    return pd.read_csv(file_path)                                                                     # Default read fallback

def pick_superstore_csv(folder):                                                                     # Define file picker
    """Find the most likely Superstore CSV inside a folder."""
    folder = Path(folder)                                                                             # Ensure path object
    csv_files = list(folder.rglob("*.csv"))                                                           # Find all CSV files
    if not csv_files:                                                                                 # Check if list empty
        raise FileNotFoundError(f"No CSV files found in: {folder}")                                   # Raise error if no files

    preferred_names = ["superstore_cleaned.csv", "SampleSuperstore.csv", "sample_superstore.csv"]     # Target file names

    for target_name in preferred_names:                                                               # Search preferred names
        for file_path in csv_files:                                                                   # Iterate found files
            if file_path.name.lower() == target_name.lower():                                         # Match file name
                return file_path                                                                      # Return matching path

    for file_path in csv_files:                                                                       # Search by keyword
        if "superstore" in file_path.name.lower():                                                    # Check if keyword exists
            return file_path                                                                          # Return keyword match

    return csv_files[0]                                                                               # Return first CSV found

if DATA_SOURCE == "preferred":                                                                        # If preferred source
    file_path = Path("superstore_cleaned.csv")                                                        # Set target path
    if not file_path.exists():                                                                        # Check if exists
        raise FileNotFoundError("Cleaned file not found.")                                            # Error if missing
    df = read_csv_robust(file_path)                                                                   # Load local data
    source_label = str(file_path)                                                                     # Set source label

elif DATA_SOURCE == "manual_upload":                                                                  # If manual upload
    from google.colab import files                                                                    # Import upload tool
    uploaded = files.upload()                                                                         # Trigger upload prompt
    csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]                   # Find uploaded CSVs
    file_path = Path(csv_files[0])                                                                    # Get first file path
    df = read_csv_robust(file_path)                                                                   # Load uploaded data
    source_label = str(file_path)                                                                     # Set source label

elif DATA_SOURCE == "kaggle":                                                                         # If Kaggle source
    import kagglehub                                                                                  # Import Kaggle API
    path = kagglehub.dataset_download("vivek468/superstore-dataset-final")                            # Download dataset
    file_path = pick_superstore_csv(path)                                                             # Pick best CSV file
    df = read_csv_robust(file_path)                                                                   # Load dataset data
    source_label = str(file_path)                                                                     # Set source label

for col in df.columns:                                                                                # Scan all columns
    if "date" in col.lower():                                                                         # Check for date keyword
        df[col] = pd.to_datetime(df[col], errors="coerce")                                            # Convert to datetime

print(f"Loaded from: {source_label}")                                                                 # Print data source
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")                                            # Print data dimensions
df.head()                                                                                             # Display first rows

In [ ]:
def first_match(columns, include=None, exclude=None, exact=None):                 # Define column matching function
    include = include or []                                                        # Default include to empty list
    exclude = exclude or []                                                        # Default exclude to empty list
    cols = list(columns)                                                           # Convert columns to list

    if exact:                                                                      # Check for exact match first
        for col in cols:                                                           # Iterate through column names
            if col.lower() == exact.lower():                                       # Case-insensitive exact comparison
                return col                                                         # Return matching column name

    for col in cols:                                                               # Iterate through columns for keyword search
        low = col.lower()                                                          # Lowercase current column name
        if all(word in low for word in include) and not any(word in low for word in exclude): # Check include/exclude words
            return col                                                             # Return first matching column
    return None                                                                    # Return None if no match found

sales_col = first_match(df.columns, include=["sales"])                             # Detect Sales column
profit_col = first_match(df.columns, include=["profit"])                           # Detect Profit column
quantity_col = first_match(df.columns, include=["quantity"])                       # Detect Quantity column
discount_col = first_match(df.columns, include=["discount"])                       # Detect Discount column
category_col = first_match(df.columns, exact="Category") or first_match(df.columns, include=["category"], exclude=["sub"]) # Detect Category
region_col = first_match(df.columns, exact="Region") or first_match(df.columns, include=["region"]) # Detect Region
order_date_col = first_match(df.columns, include=["order", "date"])                # Detect Order Date column

print("Detected columns:")                                                         # Header for printed output
print("sales_col   =", sales_col)                                                  # Print identified Sales column
print("profit_col  =", profit_col)                                                 # Print identified Profit column
print("quantity_col=", quantity_col)                                               # Print identified Quantity column
print("discount_col=", discount_col)                                               # Print identified Discount column
print("category_col=", category_col)                                               # Print identified Category column
print("region_col  =", region_col)                                                 # Print identified Region column
print("order_date  =", order_date_col)                                             # Print identified Order Date column

## Quick chart selector

| If your question is about... | Best chart |
|---|---|
| distribution of one variable | histogram |
| comparison across groups | bar chart |
| relationship between two numeric variables | scatter plot |
| spread / quartiles / outliers | boxplot |
| many numeric relationships at once | heatmap |
| change over time | line chart |


## 1) Histogram — Distribution of Sales

A **histogram** shows how values are distributed.

Look for:
- skewness
- spread
- where most observations are concentrated
- whether mean and median are far apart


In [ ]:
fig, ax = plt.subplots()                                                             # Create figure and axis objects
ax.hist(df[sales_col].dropna(), bins=40, color="steelblue", edgecolor="black", alpha=0.8) # Plot histogram of sales data

sales_mean = df[sales_col].mean()                                                       # Calculate mean of sales
sales_median = df[sales_col].median()                                                   # Calculate median of sales
ax.axvline(sales_mean, color="red", linestyle="--", label=f"Mean = {sales_mean:.2f}")       # Add vertical line for mean
ax.axvline(sales_median, color="orange", linestyle="--", label=f"Median = {sales_median:.2f}") # Add vertical line for median

ax.set_title("Distribution of Sales")                                                   # Set plot title
ax.set_xlabel("Sales ($)")                                                              # Set x-axis label
ax.set_ylabel("Frequency")                                                              # Set y-axis label
ax.legend()                                                                             # Show legend for lines
plt.show()                                                                              # Render the final plot

**Tiny TODO:**  
Change `bins=40` to `bins=20` and then to `bins=60`.  
Which version makes the shape easier to read?


**Interpretation prompt:**  
Write 1–2 sentences below:
- Is Sales symmetric, right-skewed, or left-skewed?
- Is the mean larger than the median? What does that suggest?


## 2) Bar charts — Compare groups

A **bar chart** is good when we want to compare totals or counts across categories.


In [ ]:
cat_sales = df.groupby(category_col)[sales_col].sum().sort_values(ascending=False)   # Group sales by category and sort

fig, ax = plt.subplots(figsize=(8, 5))                                              # Create figure and axis objects
cat_sales.plot(kind="bar", ax=ax, color=["#2196F3", "#4CAF50", "#FF9800"])          # Plot bar chart with specific colors
ax.set_title("Total Sales by Category")                                             # Set the title of the plot
ax.set_xlabel("Category")                                                            # Set the x-axis label
ax.set_ylabel("Total Sales ($)")                                                     # Set the y-axis label
ax.tick_params(axis="x", rotation=0)                                                 # Set x-axis labels to horizontal
plt.show()                                                                           # Render the final visualization

In [ ]:
region_sales = df.groupby(region_col)[sales_col].sum().sort_values(ascending=False) # Group sales by region and sort

fig, ax = plt.subplots(figsize=(8, 5))                                              # Create figure and axis objects
region_sales.plot(kind="bar", ax=ax, color="steelblue")                             # Plot bar chart for regions
ax.set_title("Total Sales by Region")                                               # Set the title of the plot
ax.set_xlabel("Region")                                                              # Set the x-axis label
ax.set_ylabel("Total Sales ($)")                                                     # Set the y-axis label
ax.tick_params(axis="x", rotation=0)                                                 # Set x-axis labels to horizontal
plt.show()                                                                           # Render the final visualization

**Interpretation prompt:**  
Write 2 short sentences:
- Which category has the highest total sales?
- Which region has the highest total sales?


## 3) Scatter plot — Relationship between Sales and Profit

A **scatter plot** helps us check whether two numeric variables move together, form clusters, or contain odd points.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))                                     # Create figure and axis objects
sns.scatterplot(data=df, x=sales_col, y=profit_col, ax=ax, alpha=0.6, s=35) # Plot relationship between sales and profit
ax.axhline(0, color="black", linestyle="--", linewidth=1)                 # Add horizontal line at zero profit
ax.set_title("Sales vs Profit")                                              # Set the title of the plot
ax.set_xlabel("Sales ($)")                                                   # Set the x-axis label
ax.set_ylabel("Profit ($)")                                                  # Set the y-axis label
plt.show()                                                                    # Render the final visualization

**Tiny TODO:**  
Try one improvement:
- use `hue=category_col` inside `sns.scatterplot(...)`, or
- reduce marker size, or
- increase transparency with smaller `alpha`


**Interpretation prompt:**  
Write 1–2 sentences:
- Do higher sales always mean higher profit?
- Do you notice any negative-profit points?


## 4) Boxplot + IQR — Outliers in Profit

A **boxplot** shows:
- median
- Q1 and Q3
- whiskers
- outliers

Then we use the **IQR method** numerically.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))                                         # Create figure and axis objects
sns.boxplot(x=df[profit_col], color="lightcoral", ax=ax)                         # Plot boxplot for profit data
ax.set_title("Boxplot of Profit")                                                # Set the title of the plot
ax.set_xlabel("Profit ($)")                                                      # Set the x-axis label
plt.show()                                                                       # Render the boxplot visualization

Q1 = df[profit_col].quantile(0.25)                                               # Calculate the first quartile (25%)
Q3 = df[profit_col].quantile(0.75)                                               # Calculate the third quartile (75%)
IQR = Q3 - Q1                                                                    # Calculate Interquartile Range
lower_bound = Q1 - 1.5 * IQR                                                     # Define lower bound for outliers
upper_bound = Q3 + 1.5 * IQR                                                     # Define upper bound for outliers

outliers = df[(df[profit_col] < lower_bound) | (df[profit_col] > upper_bound)]   # Filter rows outside the bounds

print(f"Q1 = {Q1:.2f}")                                                          # Print the 25th percentile value
print(f"Q3 = {Q3:.2f}")                                                          # Print the 75th percentile value
print(f"IQR = {IQR:.2f}")                                                         # Print the calculated IQR
print(f"Lower bound = {lower_bound:.2f}")                                        # Print the calculated lower limit
print(f"Upper bound = {upper_bound:.2f}")                                        # Print the calculated upper limit
print(f"Outliers found = {len(outliers)} rows ({len(outliers)/len(df)*100:.1f}%)") # Print outlier count and percentage

**Interpretation prompt:**  
Write 1–2 sentences:
- Are the outliers mostly very high profit or large losses?
- Should we automatically remove them? Why or why not?


## 5) Correlation heatmap

A **heatmap** shows many pairwise correlations at once.  
Remember: **correlation is not causation**.


In [ ]:
num_cols = [c for c in [sales_col, quantity_col, discount_col, profit_col] if c is not None] # Filter existing numeric columns
corr_matrix = df[num_cols].corr()                                                  # Calculate correlation matrix

fig, ax = plt.subplots(figsize=(7, 5))                                              # Create figure and axis objects
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)    # Plot heatmap with annotations
ax.set_title("Correlation Heatmap")                                                 # Set the title of the plot
plt.show()                                                                          # Render the final visualization

corr_matrix                                                                         # Display the correlation table

**Interpretation prompt:**  
Write 2 short sentences:
- Which pair has the strongest positive correlation?
- Is there a negative relationship? What might it suggest?


## 6) Monthly line chart — Trend over time

A **line chart** is best when the x-axis is time and we want to see trend / seasonality / spikes.


In [ ]:
monthly_sales = df.resample("M", on=order_date_col)[sales_col].sum()   # Resample data by month and sum sales

fig, ax = plt.subplots(figsize=(12, 5))                                  # Create figure and axis objects
monthly_sales.plot(ax=ax, color="steelblue", linewidth=2)                 # Plot monthly sales as a line chart
ax.set_title("Monthly Total Sales Over Time")                            # Set the title of the plot
ax.set_xlabel("Month")                                                   # Set the x-axis label
ax.set_ylabel("Total Sales ($)")                                         # Set the y-axis label
ax.grid(True, alpha=0.3)                                                 # Add a light grid to the plot
plt.show()                                                               # Render the final visualization

**Interpretation prompt:**  
Write 1–2 sentences:
- Do you see peaks or dips?
- Is there any possible seasonality?


## 7) Mini challenge

Create **one extra plot** of your own choice:
- Sales by Segment
- Profit by Region
- Discount distribution
- Another scatter plot with color by category


In [ ]:
# Your extra plot here


## Final reflection

Answer briefly:

1. Why is it dangerous to present a chart without labels?
2. Why can the same dataset look different depending on chart type?
3. Which chart from today would be most useful for a store manager, and why?
